# 04b — LoRA Fine-tuning + Post-LoRA Attribution

**Runs on RunPod GPU (RTX 4090, 24 GB).**

This notebook does two things in sequence:

1. **Fine-tunes** Gemma-2-2B with a PEFT LoRA adapter on `lora_train_combined.jsonl`
   (generated by `04_lora_training_data.ipynb`). Loss is computed only on the
   completion token (` True` / ` False`).  Adapter is saved to
   `my_work/checkpoints/lora_combined/`.

2. **Re-runs attribution** on a stratified **pilot subset** (~40 prompts) of the
   frozen probe set (`prompts_triangle_v2.jsonl`) using the LoRA-merged model.
   If the pilot succeeds, a flag cell lets you scale to the full 270 binary
   prompts.  Stats are saved to
   `my_work/results/statistics/stats_lora_combined_pilot.json`
   and (after scale-up) `stats_lora_combined.json`.

**Prerequisites:**
- `04_lora_training_data.ipynb` must have been run (produces `lora_train_combined.jsonl`).
- `01_dataset_generation.ipynb` must have been run (produces `prompts_triangle_v2.jsonl`).
- `pip install peft transformers accelerate` (below).

**Key constraints (do not change):**
- `prompts_triangle_v2.jsonl` is **never** modified.
- Transcoder stays as the **base** CLT (option (a): apples-to-apples comparison).
- Attribution uses **identical** `ATTR_KWARGS` and `compute_statistics` as `02`.

## 0 — Environment setup

In [ ]:
import os
import sys
from pathlib import Path

# Must be set BEFORE torch is imported anywhere.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR.")

try:
    from dotenv import load_dotenv
    if _root is not None and (_root / ".env").is_file():
        load_dotenv(_root / ".env")
        print("Loaded .env")
except ImportError:
    pass

MY_WORK = _my_work if _root else Path(".").resolve()

if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    for key, path in {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
    }.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)
    print(f"RunPod HF cache: {hf_home}")

CHECKPOINT_DIR = MY_WORK / "checkpoints" / "lora_combined"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

STATS_DIR = MY_WORK / "results" / "statistics"
STATS_DIR.mkdir(parents=True, exist_ok=True)

GRAPHS_DIR = MY_WORK / "results" / "graphs_lora_combined"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"MY_WORK      : {MY_WORK}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")
print(f"Stats dir    : {STATS_DIR}")

In [ ]:
# Install PEFT + accelerate if not present
%pip install -q peft accelerate

## 1 — Constants

In [ ]:
import json
import time
import gc
import traceback
import random

import torch

# ── Model & tokenizer constants ────────────────────────────────────────────────
MODEL_NAME      = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"  # base CLT — unchanged from baseline
TOKEN_TRUE      = " True"
TOKEN_FALSE     = " False"
VOCAB_ID_TRUE   = 5569
VOCAB_ID_FALSE  = 7662

# ── LoRA hyperparameters ───────────────────────────────────────────────────────
LORA_R          = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "gate_proj", "up_proj"]

# ── Training hyperparameters ───────────────────────────────────────────────────
TRAIN_LR        = 2e-4
TRAIN_BATCH     = 4       # per-device; increase to 8 if VRAM allows
TRAIN_MAX_STEPS = 400     # ~400 steps on ~600 rows ≈ 2-3 epochs
TRAIN_WARMUP    = 40      # 10% warmup
MAX_SEQ_LEN     = 128     # prompts are short; cap to save memory
SEED            = 42

# ── Attribution constants (identical to 02) ───────────────────────────────────
ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=False,
)
PHASE = "lora_combined"

# ── Pilot vs full ─────────────────────────────────────────────────────────────
# Set RUN_FULL=True to run all 270 binary prompts after the pilot succeeds.
# Start with RUN_FULL=False to validate the pipeline on ~40 prompts first.
RUN_FULL = False
N_PILOT  = 40     # stratified sample; see section 4

PILOT_STATS_FILE = STATS_DIR / "stats_lora_combined_pilot.json"
FULL_STATS_FILE  = STATS_DIR / "stats_lora_combined.json"
STATS_FILE = FULL_STATS_FILE if RUN_FULL else PILOT_STATS_FILE

# ── Paths ──────────────────────────────────────────────────────────────────────
TRAIN_JSONL  = MY_WORK / "data" / "lora_train_combined.jsonl"
PROBE_JSONL  = MY_WORK / "data" / "prompts_triangle_v2.jsonl"

print(f"Model        : {MODEL_NAME}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}, target={LORA_TARGET_MODULES}")
print(f"Training     : lr={TRAIN_LR}, steps={TRAIN_MAX_STEPS}, batch={TRAIN_BATCH}")
print(f"RUN_FULL     : {RUN_FULL}  (pilot N={N_PILOT})")
print(f"Stats output : {STATS_FILE}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

## 2 — Load training data

In [ ]:
train_rows = []
with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line.strip())
        if row.get("completion") in (" True", " False"):
            train_rows.append(row)

n_true  = sum(1 for r in train_rows if r["label"])
n_false = sum(1 for r in train_rows if not r["label"])
l1_n    = sum(1 for r in train_rows if r.get("dataset_id") == "L1")
l2_n    = sum(1 for r in train_rows if r.get("dataset_id") == "L2")

print(f"Training rows loaded: {len(train_rows)}")
print(f"  True : {n_true}  |  False: {n_false}")
print(f"  L1   : {l1_n}   |  L2   : {l2_n}")
print()
print(f"Probe JSONL  : {PROBE_JSONL}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")

## 3 — Load HuggingFace model for LoRA training

We load the raw HuggingFace `AutoModelForCausalLM` (not the circuit-tracer
`ReplacementModel`) for training.  After training, the adapter is saved.
For attribution, we reload the circuit-tracer `ReplacementModel` and merge
the LoRA weights into it.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

device_str = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_str)

print(f"Loading tokenizer from {MODEL_NAME} ...")
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if hf_tokenizer.pad_token is None:
    hf_tokenizer.pad_token = hf_tokenizer.eos_token
hf_tokenizer.padding_side = "left"  # Gemma uses left-padding for generation

print(f"Loading base model from {MODEL_NAME} ...")
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("Wrapping with LoRA ...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
lora_model = get_peft_model(hf_model, lora_config)
lora_model.print_trainable_parameters()

## 4 — Build tokenised dataset for training

In [ ]:
from torch.utils.data import Dataset, DataLoader


class CompletionDataset(Dataset):
    """Tokenises (prompt, completion) pairs.  Loss is masked to completion only."""

    def __init__(self, rows, tokenizer, max_len):
        self.samples = []
        for row in rows:
            full_text = row["prompt"] + row["completion"]
            enc_full   = tokenizer(full_text,   truncation=True, max_length=max_len)
            enc_prompt = tokenizer(row["prompt"], truncation=True, max_length=max_len)
            n_prompt = len(enc_prompt["input_ids"])
            input_ids = enc_full["input_ids"]
            labels    = [-100] * n_prompt + input_ids[n_prompt:]
            if len(labels) < len(input_ids):
                labels = [-100] * len(input_ids)
            labels = labels[:len(input_ids)]
            self.samples.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels":    torch.tensor(labels,    dtype=torch.long),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    """Left-pad input_ids and labels to the longest sequence in the batch."""
    max_len = max(len(b["input_ids"]) for b in batch)
    padded_ids    = []
    padded_labels = []
    attn_masks    = []
    pad_id = hf_tokenizer.pad_token_id
    for b in batch:
        ids = b["input_ids"]
        lbl = b["labels"]
        pad_len = max_len - len(ids)
        padded_ids.append(torch.cat([torch.full((pad_len,), pad_id, dtype=torch.long), ids]))
        padded_labels.append(torch.cat([torch.full((pad_len,), -100, dtype=torch.long), lbl]))
        attn_masks.append(torch.cat([torch.zeros(pad_len, dtype=torch.long), torch.ones(len(ids), dtype=torch.long)]))
    return {
        "input_ids":      torch.stack(padded_ids),
        "labels":         torch.stack(padded_labels),
        "attention_mask": torch.stack(attn_masks),
    }


random.seed(SEED)
shuffled_rows = list(train_rows)
random.shuffle(shuffled_rows)

train_dataset = CompletionDataset(shuffled_rows, hf_tokenizer, MAX_SEQ_LEN)
train_loader  = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH,
    shuffle=True,
    collate_fn=collate_fn,
)

print(f"Dataset samples : {len(train_dataset)}")
print(f"Batches per step: {len(train_loader)} (batch_size={TRAIN_BATCH})")  
print(f"Target steps    : {TRAIN_MAX_STEPS}")

# Sanity: check completion token IDs
id_true  = hf_tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = hf_tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"Token mismatch: True  {id_true} != {VOCAB_ID_TRUE}"
assert id_false == VOCAB_ID_FALSE, f"Token mismatch: False {id_false} != {VOCAB_ID_FALSE}"
print(f"Token IDs verified: True={id_true}, False={id_false}")

## 5 — Training loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR

optimizer = AdamW(lora_model.parameters(), lr=TRAIN_LR, weight_decay=0.01)

# Linear warmup then linear decay
warmup_scheduler = LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=TRAIN_WARMUP
)
decay_scheduler = LinearLR(
    optimizer, start_factor=1.0, end_factor=0.1, total_iters=TRAIN_MAX_STEPS - TRAIN_WARMUP
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, decay_scheduler],
    milestones=[TRAIN_WARMUP],
)

lora_model.train()
global_step = 0
running_loss = 0.0
log_every = 20
t0_train = time.time()

print(f"Starting training: {TRAIN_MAX_STEPS} steps, lr={TRAIN_LR}")
print()

loader_iter = iter(train_loader)

while global_step < TRAIN_MAX_STEPS:
    try:
        batch = next(loader_iter)
    except StopIteration:
        loader_iter = iter(train_loader)  # restart epoch
        batch = next(loader_iter)

    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()

    outputs = lora_model(**batch)
    loss = outputs.loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(lora_model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    running_loss += loss.item()
    global_step  += 1

    if global_step % log_every == 0 or global_step == TRAIN_MAX_STEPS:
        avg_loss = running_loss / log_every
        lr_now   = scheduler.get_last_lr()[0]
        elapsed  = time.time() - t0_train
        print(
            f"step {global_step:4d}/{TRAIN_MAX_STEPS}  "
            f"loss={avg_loss:.4f}  lr={lr_now:.2e}  "
            f"elapsed={elapsed:.0f}s"
        )
        running_loss = 0.0

print()
print(f"Training complete in {time.time() - t0_train:.0f}s.")

## 6 — Save LoRA adapter

In [ ]:
lora_model.save_pretrained(str(CHECKPOINT_DIR))
hf_tokenizer.save_pretrained(str(CHECKPOINT_DIR))

print(f"LoRA adapter saved to: {CHECKPOINT_DIR}")
print(f"  Files: {[f.name for f in CHECKPOINT_DIR.iterdir()]}")

## 7 — Load circuit-tracer ReplacementModel with merged LoRA weights

The circuit-tracer `ReplacementModel` wraps a `HookedTransformer` and loads the
base CLT transcoder.  We merge the LoRA weights into the HuggingFace model,
save a temporary merged checkpoint, then load it via `ReplacementModel`.

This keeps the **base CLT** unchanged (option a) and uses the exact same
`attribute()` path as notebook `02`.

In [ ]:
import tempfile
from peft import PeftModel
from circuit_tracer import ReplacementModel

# Free the PEFT training model from GPU first
lora_model.to("cpu")
del lora_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Reload base + adapter, merge, and save to a temp dir
print("Merging LoRA weights into base model ...")
MERGED_DIR = MY_WORK / "checkpoints" / "lora_combined_merged"
MERGED_DIR.mkdir(parents=True, exist_ok=True)

_base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cpu"
)
_peft_for_merge = PeftModel.from_pretrained(_base_for_merge, str(CHECKPOINT_DIR))
_merged = _peft_for_merge.merge_and_unload()
_merged.save_pretrained(str(MERGED_DIR))
hf_tokenizer.save_pretrained(str(MERGED_DIR))

del _base_for_merge, _peft_for_merge, _merged
gc.collect()

print(f"Merged checkpoint saved to: {MERGED_DIR}")

# Now load via circuit-tracer using the merged checkpoint
print("Loading ReplacementModel from merged checkpoint ...")
device = torch.device(device_str)
ct_model = ReplacementModel.from_pretrained(
    str(MERGED_DIR),
    TRANSCODER_NAME,
    dtype=torch.bfloat16,
    backend="transformerlens",
    device=device,
    lazy_encoder=True,
    lazy_decoder=True,
)
ct_tokenizer = ct_model.tokenizer

# Verify token IDs
id_true  = ct_tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = ct_tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"Token mismatch True:  {id_true}"
assert id_false == VOCAB_ID_FALSE, f"Token mismatch False: {id_false}"
print(f"ReplacementModel loaded. Token IDs verified: True={id_true}, False={id_false}")

## 8 — Load probe set + select pilot subset

In [ ]:
from collections import Counter

all_probe = []
with open(PROBE_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line.strip())
        if row.get("task_type", "binary") == "binary":
            all_probe.append(row)

print(f"Binary probe prompts: {len(all_probe)}")
print()

if RUN_FULL:
    probe_prompts = all_probe
    print(f"RUN_FULL=True: running all {len(probe_prompts)} binary prompts.")
else:
    # Stratified pilot sample across (family, tail, label)
    from collections import defaultdict
    strata: dict = defaultdict(list)
    for p in all_probe:
        key = (p.get("family", "?"), p.get("tail", "?"), str(p.get("label", "?")))
        strata[key].append(p)

    rng = random.Random(SEED)
    pilot = []
    n_strata = len(strata)
    per_stratum = max(1, N_PILOT // n_strata)

    for key, bucket in sorted(strata.items()):
        rng.shuffle(bucket)
        pilot.extend(bucket[:per_stratum])

    # Top up to N_PILOT if needed (take remaining from largest strata)
    remaining_pool = [p for p in all_probe if p not in pilot]
    rng.shuffle(remaining_pool)
    while len(pilot) < N_PILOT and remaining_pool:
        pilot.append(remaining_pool.pop(0))

    probe_prompts = pilot[:N_PILOT]
    print(f"RUN_FULL=False: pilot subset = {len(probe_prompts)} prompts")
    dist = Counter(
        (p.get("family", "?"), p.get("tail", "?"), str(p.get("label", "?")))
        for p in probe_prompts
    )
    for key, cnt in sorted(dist.items()):
        print(f"  {key}: {cnt}")

## 9 — First-token accuracy check (post-LoRA, pre-attribution)

In [ ]:
correct = 0
true_correct  = 0; true_total  = 0
false_correct = 0; false_total = 0
predictions = []

with torch.inference_mode():
    for entry in probe_prompts:
        logits, _ = ct_model.feature_intervention(entry["prompt"], [])
        pred_id = int(logits.squeeze(0)[-1].argmax().item())
        pred_tok = ct_tokenizer.decode([pred_id])
        expected_id = VOCAB_ID_TRUE if entry["label"] else VOCAB_ID_FALSE
        is_ok = (pred_id == expected_id)
        if is_ok:
            correct += 1
        if entry["label"]:
            true_total += 1
            if pred_id == VOCAB_ID_TRUE:
                true_correct += 1
        else:
            false_total += 1
            if pred_id == VOCAB_ID_FALSE:
                false_correct += 1
        predictions.append({
            "prompt_id": entry["prompt_id"],
            "label": entry["label"],
            "pred_token": pred_tok,
            "pred_id": pred_id,
            "is_correct": is_ok,
        })

n = len(probe_prompts)
print(f"Post-LoRA first-token accuracy on probe subset (n={n}):")
print(f"  Overall : {correct}/{n} = {correct/n:.1%}")
if true_total:
    print(f"  True    : {true_correct}/{true_total} = {true_correct/true_total:.1%}")
if false_total:
    print(f"  False   : {false_correct}/{false_total} = {false_correct/false_total:.1%}")
print()
print("Note: base model accuracy was ~18.5% on binary prompts.")
print("An increase here confirms the LoRA successfully changed the output distribution.")

## 10 — Attribution + stats loop (post-LoRA)

Identical to notebook `02`, using the LoRA-merged `ct_model`.
Stats schema is identical — join on `prompt_id` in `05_lora_comparison.ipynb`.

In [ ]:
import importlib
import utils.graph_statistics as gs_mod
importlib.reload(gs_mod)

from circuit_tracer import attribute
from utils.graph_statistics import compute_statistics, load_statistics, append_statistic

# Resume: skip already-succeeded prompts
existing = load_statistics(STATS_FILE)
done_ids = {e["prompt_id"] for e in existing if e.get("attribution_succeeded")}
n_prev_failed = sum(1 for e in existing if not e.get("attribution_succeeded"))
remaining = [p for p in probe_prompts if p["prompt_id"] not in done_ids]

print(f"Stats file     : {STATS_FILE}")
print(f"Already done   : {len(done_ids)} succeeded")
print(f"Previously failed: {n_prev_failed} (will retry)")
print(f"Remaining      : {len(remaining)} prompts")
print()

t0 = time.time()
n_success = 0
n_fail = 0
progress_every = 5

for i, entry in enumerate(remaining, start=1):
    pid = entry["prompt_id"]
    print(f"[{i}/{len(remaining)}] {pid} ...", end=" ", flush=True)

    try:
        graph = attribute(
            prompt=entry["prompt"],
            model=ct_model,
            attribution_targets=[TOKEN_TRUE, TOKEN_FALSE],
            **ATTR_KWARGS,
        )

        stat = compute_statistics(graph, entry, phase=PHASE)
        append_statistic(stat, STATS_FILE)
        n_success += 1

        del graph
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        density = stat.get("edge_density", float("nan"))
        n_active = stat.get("n_active_features", "?")
        print(f"OK  n_active={n_active}  density={density:.4f}")

    except Exception as exc:
        n_fail += 1
        stat = {
            "prompt_id": pid,
            "attribution_succeeded": False,
            "error": str(exc)[:200],
            "phase": PHASE,
        }
        append_statistic(stat, STATS_FILE)
        print(f"FAIL: {exc}")
        traceback.print_exc()

    if i % progress_every == 0:
        elapsed = time.time() - t0
        speed   = i / elapsed
        eta     = (len(remaining) - i) / speed if speed > 0 else float("inf")
        print(
            f"[progress] {i}/{len(remaining)} ({i/len(remaining):.0%}) | "
            f"success={n_success/(n_success+n_fail):.0%} | "
            f"{speed:.3f} prompts/s | ETA {eta/60:.1f} min"
        )

print()
elapsed_total = time.time() - t0
print(f"Done. {n_success} succeeded, {n_fail} failed in {elapsed_total:.0f}s.")
print(f"Stats saved to: {STATS_FILE}")

## 11 — Scale-up decision

After the pilot run, review the results. If `n_success >= N_PILOT * 0.9`
and the stats look reasonable (no all-NaN columns, no immediate OOM), set
`RUN_FULL = True` in section 1 and **re-run from section 8 onwards** to
run all 270 binary prompts.

The resume logic in section 10 will skip already-done IDs.

In [ ]:
final_stats = load_statistics(STATS_FILE)
n_ok = sum(1 for s in final_stats if s.get("attribution_succeeded"))
n_total = len(final_stats)

print(f"=== Post-run summary ===")
print(f"Stats file : {STATS_FILE}")
print(f"Total rows : {n_total}")
print(f"Succeeded  : {n_ok}  ({n_ok/max(n_total,1):.1%})")
print()
if not RUN_FULL:
    print("Pilot complete.")
    print("If results look good, set RUN_FULL=True in section 1 and re-run from section 8.")
    print("Full run will save stats to:", FULL_STATS_FILE)
else:
    print("Full run complete. Proceed to 05_lora_comparison.ipynb.")